In [ ]:
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [ ]:
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

# Lab 1: Retrieval & Document-Processing Agents

Build the first two CareConnect specialists with Strands, pointed at the `-sdk`
Knowledge Base from Lab 0. These mirror `retrieval_agent.py` and
`document_processing_agent.py` from your console build, but run in-notebook.

### Step 1: Imports

In [ ]:
import lab_helpers.utils as u
from lab_helpers.careconnect_agents import build_retrieval_agent, build_docproc_agent
print("KB in use:", u.get_ssm_parameter(f"{u.SSM_PREFIX}/kb_id"))

### Step 2: The Retrieval Agent

In [ ]:
retrieval_agent = build_retrieval_agent()
resp = retrieval_agent("How should a patient prepare for a colonoscopy?")
print(resp)

### Step 3: The Document-Processing Agent

It only *structures* retrieved evidence — it never adds new medical content.

In [ ]:
docproc_agent = build_docproc_agent()

# Feed the retrieved passages from the retrieval agent as DATA.
evidence = str(resp)
structured = docproc_agent(
    "Convert the following approved evidence into a clear numbered checklist. "
    "Use ONLY what is in the evidence.\n\n<evidence>\n" + evidence + "\n</evidence>")
print(structured)

### Step 4: Negative test — no invented content

In [ ]:
empty = docproc_agent(
    "Structure this evidence:\n<evidence>\nNo approved Riverside Health "
    "information was found for this question.\n</evidence>")
print(empty)  # should decline to invent instructions

## Lab 1 complete ✅

Retrieval and Document-Processing agents work against the `-sdk` KB.